In [30]:
import numpy as np
import matplotlib.pylab as pl
import plotly.express as px
import plotly.graph_objects as go
from tqdm import tqdm


from sklearn.datasets import make_multilabel_classification
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [31]:
n_rep = 100
n_samples = 1000
n_classes = 10
n_labels = 5

control_level = 1 / 10
print(control_level)

regularization_grid = np.logspace(-3, 1, 50)
print(regularization_grid)

classifiers = [
    OneVsRestClassifier(LogisticRegression(C=regularization))
    for regularization in regularization_grid
]

0.1
[1.00000000e-03 1.20679264e-03 1.45634848e-03 1.75751062e-03
 2.12095089e-03 2.55954792e-03 3.08884360e-03 3.72759372e-03
 4.49843267e-03 5.42867544e-03 6.55128557e-03 7.90604321e-03
 9.54095476e-03 1.15139540e-02 1.38949549e-02 1.67683294e-02
 2.02358965e-02 2.44205309e-02 2.94705170e-02 3.55648031e-02
 4.29193426e-02 5.17947468e-02 6.25055193e-02 7.54312006e-02
 9.10298178e-02 1.09854114e-01 1.32571137e-01 1.59985872e-01
 1.93069773e-01 2.32995181e-01 2.81176870e-01 3.39322177e-01
 4.09491506e-01 4.94171336e-01 5.96362332e-01 7.19685673e-01
 8.68511374e-01 1.04811313e+00 1.26485522e+00 1.52641797e+00
 1.84206997e+00 2.22299648e+00 2.68269580e+00 3.23745754e+00
 3.90693994e+00 4.71486636e+00 5.68986603e+00 6.86648845e+00
 8.28642773e+00 1.00000000e+01]


In [32]:
def generate_data_set(n_samples, n_classes, n_labels):
    inputs, outputs = make_multilabel_classification(
        n_samples=n_samples,
        n_classes=n_classes,
        n_labels=n_labels,
        allow_unlabeled=False,
        return_indicator=True,
    )
    input_test, output_test = make_multilabel_classification(
        n_samples=1,
        n_classes=5,
        n_labels=3,
        allow_unlabeled=False,
        return_indicator=True,
    )

    inputs_train, inputs_calibration, outputs_train, outputs_calibration = (
        train_test_split(inputs, outputs, test_size=0.5, random_state=42)
    )

    input_scaler = StandardScaler()
    scaled_inputs_train = input_scaler.fit_transform(inputs_train)
    scaled_inputs_calibration = input_scaler.transform(inputs_calibration)
    scaled_input_test = input_scaler.transform(input_test)

    return (
        scaled_inputs_train,
        scaled_inputs_calibration,
        scaled_input_test,
        outputs_train,
        outputs_calibration,
        output_test,
    )

In [33]:
def make_FNR_risk(outputs_calibration, probas):
    sample_size = outputs_calibration.shape[0]

    def FNR_risk(q_value):
        return (
            (np.logical_and(outputs_calibration, probas < 1 - q_value)).sum(axis=1)
            / outputs_calibration.sum(axis=1)
        ).mean()

    def upper_FNR_risk(q_value):
        return (1.0 + sample_size * FNR_risk(q_value)) / (sample_size + 1)

    def lower_FNR_risk(q_value):
        return (0.0 + sample_size * FNR_risk(q_value)) / (sample_size + 1)

    return FNR_risk, upper_FNR_risk, lower_FNR_risk


def dichotomy_search(f_values):
    cardinal = f_values.shape[0]
    left_index = 0
    right_index = cardinal - 1

    f_right = f_values[right_index]
    f_left = f_values[left_index]

    if f_right * f_left > 0:
        if f_left > 0:
            return right_index
        else:
            return left_index

    while (right_index - left_index) >= 2:
        middle_index = max(
            min(np.int64((left_index + right_index) / 2), right_index), left_index
        )
        f_middle = f_values[middle_index]

        # print("left", left_index, f_left)
        # print("middle", middle_index, f_middle)
        # print("right", right_index, f_right)

        if (f_middle == 0) or (f_left * f_middle < 0):
            right_index = middle_index
            f_right = f_middle
        else:
            left_index = middle_index
            f_left = f_middle

    if f_left * f_right < 0:
        return right_index
    else:
        return left_index

In [34]:
def model_selection(
    scaled_inputs_train,
    scaled_inputs_calibration,
    scaled_input_test,
    outputs_train,
    outputs_calibration,
    classifiers,
):
    for classifier in classifiers:
        classifier.fit(scaled_inputs_train, outputs_train)

    probas_calibration = [
        classifier.predict_proba(scaled_inputs_calibration)
        for classifier in classifiers
    ]
    proba_test = [
        classifier.predict_proba(scaled_input_test) for classifier in classifiers
    ]

    q_values_ = [np.sort(probas.flatten()) for probas in probas_calibration]
    risks = [
        make_FNR_risk(outputs_calibration, probas) for probas in probas_calibration
    ]

    upper_fnr_risk_values_ = [
        np.array([FNR_risk[1](q_value) for q_value in q_values])
        for FNR_risk, q_values in zip(risks, q_values_)
    ]
    lower_fnr_risk_values_ = [
        np.array([FNR_risk[-1](q_value) for q_value in q_values])
        for FNR_risk, q_values in zip(risks, q_values_)
    ]

    lower_quantile_indices = [
        dichotomy_search(lower_fnr_risk_values - control_level)
        for lower_fnr_risk_values in lower_fnr_risk_values_
    ]

    upper_quantile_indices = [
        dichotomy_search(upper_fnr_risk_values - control_level)
        for upper_fnr_risk_values in upper_fnr_risk_values_
    ]

    lower_quantile_values = [
        q_values[lower_quantile_index]
        for q_values, lower_quantile_index in zip(q_values_, lower_quantile_indices)
    ]

    upper_quantile_values = [
        q_values[upper_quantile_index]
        for q_values, upper_quantile_index in zip(q_values_, upper_quantile_indices)
    ]

    lower_volumes = np.array(
        [
            (np.concatenate((probas, proba)) >= 1 - lower_quantile_value)
            .sum(axis=1)
            .mean()
            for probas, proba, lower_quantile_value in zip(
                probas_calibration, proba_test, lower_quantile_values
            )
        ]
    )

    upper_volumes = np.array(
        [
            (np.concatenate((probas, proba)) >= 1 - upper_quantile_values)
            .sum(axis=1)
            .mean()
            for probas, proba, upper_quantile_values in zip(
                probas_calibration, proba_test, upper_quantile_values
            )
        ]
    )

    min_upper_volumes = upper_volumes.min()

    modelSel_set = np.arange(regularization_grid.shape[0])[
        lower_volumes <= min_upper_volumes
    ]
    modelSel_prediction_set = np.concatenate(
        tuple(
            [
                proba_test[index] >= 1 - upper_quantile_values[index]
                for index in modelSel_set
            ]
        )
    ).any(axis=0)

    return modelSel_prediction_set

In [35]:
modelSel_prediction_sets = []
volumes = []
for i_rep in tqdm(range(n_rep)):
    (
        scaled_inputs_train,
        scaled_inputs_calibration,
        scaled_input_test,
        outputs_train,
        outputs_calibration,
        output_test,
    ) = generate_data_set(n_samples, n_classes, n_labels)
    modelSel_prediction_set = model_selection(
        scaled_inputs_train,
        scaled_inputs_calibration,
        scaled_input_test,
        outputs_train,
        outputs_calibration,
        classifiers,
    )
    modelSel_prediction_sets += [modelSel_prediction_set]
    volumes += [modelSel_prediction_set.sum()]
    print(np.mean(np.array(volumes)))

  1%|          | 1/100 [00:27<44:39, 27.07s/it]

6.0


  2%|▏         | 2/100 [00:54<44:09, 27.04s/it]

7.0


  3%|▎         | 3/100 [01:21<43:40, 27.01s/it]

...

7.3979591836734695


 99%|█████████▉| 99/100 [21:55<00:11, 11.39s/it]

7.393939393939394


100%|██████████| 100/100 [22:07<00:00, 13.27s/it]

7.41
